# API Logging & Labelled API-Log Dataset (Model Extraction Detection)

## 1) Setup

In [32]:
import os, json, time, zipfile, shutil
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

OUT_DIR = "/content/api_log_dataset"
os.makedirs(OUT_DIR, exist_ok=True)
SEED = 42
np.random.seed(SEED)

## 2) Load Victim Model

In [33]:
from google.colab import drive
drive.mount("/content/drive")

VICTIM_PATH = "/content/victim_model"
if not os.path.exists(f"{VICTIM_PATH}/config.json"):
    zipfile.ZipFile("/content/drive/MyDrive/bert_victim_model.zip").extractall(VICTIM_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(VICTIM_PATH)
model = AutoModelForSequenceClassification.from_pretrained(VICTIM_PATH).to(device).eval()
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

@torch.no_grad()
def predict_probs(texts):
    enc = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    return torch.softmax(model(**enc).logits, dim=1).cpu().numpy()

print("Victim loaded on", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Victim loaded on cuda


## 3) Victim API & Logging

In [34]:
class VictimAPI:
    def __init__(self, predict_probs_fn, id2label, log_path=f"{OUT_DIR}/api_logs.jsonl", return_probs=False):
        self.predict_probs = predict_probs_fn
        self.id2label = id2label
        self.return_probs = return_probs
        self.log_path = log_path
        self.logs = []
        open(self.log_path, "w").close()

    def query(self, query_id, user_id, session_id, text, request_timestamp=None):
        t_in = time.perf_counter()
        probs = self.predict_probs([text])[0]
        pred_id = int(np.argmax(probs))
        response_latency = (time.perf_counter() - t_in) * 1000

        record = {
            "query_id": query_id,
            "user_id": user_id,
            "session_id": session_id,
            "request_timestamp": (request_timestamp or datetime.now()).isoformat(),
            "input_reference": text,
            "predicted_label": self.id2label[pred_id],
            "confidence_scores": {self.id2label[i]: round(float(p), 6) for i, p in enumerate(probs)},
            "response_latency": round(response_latency, 3),
        }
        self.logs.append(record)
        with open(self.log_path, "a") as f:
            f.write(json.dumps(record) + "\n")

        resp = {"label": record["predicted_label"], "confidence": float(probs[pred_id])}
        if self.return_probs:
            resp["probabilities"] = record["confidence_scores"]
        return resp

    def logs_df(self):
        return pd.DataFrame(self.logs)

## 4) Load Datasets

In [35]:
def read_zip_csv(zip_path, csv_name):
    with zipfile.ZipFile(zip_path) as z:
        return pd.read_csv(z.open(csv_name))

normal_df = read_zip_csv("/content/normal_user.zip", "normal_queries.csv")
attacker_df = read_zip_csv("/content/attacker_query_dataset_v2.zip", "attacker_query_dataset_v2/attacker_queries.csv")

for name, d in [("normal", normal_df), ("attacker", attacker_df)]:
    print(f"{name}: {len(d)} queries | {d['user_id'].nunique()} users | {d['session_id'].nunique()} sessions")
print(attacker_df["attack_strategy"].value_counts())

normal: 3000 queries | 115 users | 407 sessions
attacker: 3000 queries | 30 users | 150 sessions
attack_strategy
synthetic_queries             300
adversarial_perturbation      300
ood_queries                   300
feature_space_exploration     300
information_seeking           300
hardness_oriented             300
adaptive_substitute_proxy     300
distributed_extraction        300
low_and_slow_extraction       300
in_distribution_extraction    300
Name: count, dtype: int64


## 5) Replay Queries Through Victim API

In [36]:
cols = ["query_id", "user_id", "session_id", "request_timestamp", "input_text"]
replay = pd.concat([normal_df[cols], attacker_df[cols]], ignore_index=True)
replay["ts"] = pd.to_datetime(replay["request_timestamp"], utc=True).dt.tz_localize(None)
replay = replay.sort_values(["ts", "query_id"]).reset_index(drop=True)

api = VictimAPI(predict_probs, id2label)
for r in tqdm(replay.itertuples(index=False), total=len(replay)):
    api.query(r.query_id, r.user_id, r.session_id, str(r.input_text), r.ts.to_pydatetime())

raw_logs = api.logs_df()
raw_logs.to_csv(f"{OUT_DIR}/raw_api_logs.csv", index=False)
print("Requests sent:", len(replay), "| Log entries:", len(raw_logs))
raw_logs.head()

  0%|          | 0/6000 [00:00<?, ?it/s]

Requests sent: 6000 | Log entries: 6000


,query_id,user_id,session_id,request_timestamp,input_reference,predicted_label,confidence_scores,response_latency
0,q_100001,client_136,session_136_01,2026-10-01T05:05:43,testing 123,Sci/Tech,"{'World': 0.000486, 'Sports': 0.000192, 'Busin...",33.467
1,q_100002,client_136,session_136_01,2026-10-01T05:06:09,aurora semiconductors model cuts energy use by...,Sci/Tech,"{'World': 0.000448, 'Sports': 4.8e-05, 'Busine...",26.280
2,q_100003,client_125,session_125_01,2026-10-01T06:59:06,A space probe launched by Cobalt Technologies ...,Sci/Tech,"{'World': 0.003395, 'Sports': 0.000128, 'Busin...",26.771
3,q_100004,client_125,session_125_01,2026-10-01T06:59:51,The president of Spain will travel to Poland w...,World,"{'World': 0.999099, 'Sports': 4.3e-05, 'Busine...",28.154
4,q_100005,client_125,session_125_01,2026-10-01T07:00:27,is it true that halden retail recalled 40000 u...,Business,"{'World': 0.000817, 'Sports': 0.00031, 'Busine...",25.883


## 6) Labelled Logs

In [37]:
ground_truth = pd.concat([normal_df, attacker_df], ignore_index=True)[["query_id", "label", "attack_strategy"]]
labelled_logs = raw_logs.merge(ground_truth, on="query_id", how="left", validate="one_to_one")
labelled_logs.to_csv(f"{OUT_DIR}/labelled_api_logs.csv", index=False)

print("Unmatched log entries:", labelled_logs["label"].isna().sum())
print(labelled_logs.groupby("label")["user_id"].nunique())
print(labelled_logs["attack_strategy"].value_counts())

Unmatched log entries: 0
label
attacker     30
normal      115
Name: user_id, dtype: int64
attack_strategy
none                          3000
synthetic_queries              300
adversarial_perturbation       300
ood_queries                    300
feature_space_exploration      300
information_seeking            300
hardness_oriented              300
adaptive_substitute_proxy      300
distributed_extraction         300
low_and_slow_extraction        300
in_distribution_extraction     300
Name: count, dtype: int64


## 7) Download

In [38]:
from google.colab import files
shutil.make_archive("/content/api_log_dataset", "zip", OUT_DIR)
files.download("/content/api_log_dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>